# 06 — Analysis & Figures

Reads `results/raw_predictions/{model}/S*.json` and computes:
- Per-task accuracy table (S0–S5)
- Confidence summaries (verbalized + logprob-based)
- ECE per stage
- Confidence Drop (ΔConf) under modality ablation
- Attribution Faithfulness Score (AFS) — the headline metric
- Transcript Injection Bias (TIB)
- Lexical Override Rate (LOR)
- Answer flip rates between stage pairs

Outputs land in `results/metrics/` (JSONs) and `results/figures/` (PNGs).


In [ ]:
import os, sys, json
REPO = '/content/omnimodel-research'
if os.path.exists(REPO):
    %cd $REPO
    sys.path.insert(0, REPO)

import matplotlib.pyplot as plt
import numpy as np

from src.metrics import (
    accuracy_per_task, verbalized_confidence_stats, logprob_confidence_stats,
    expected_calibration_error, confidence_drop, attribution_faithfulness,
    transcript_injection_bias, lexical_override_rate, answer_flip_rate,
)


In [ ]:
MODEL = 'qwen'   # change to 'gemma' to analyze the cross-model run
PRED_DIR = f'results/raw_predictions/{MODEL}'
METRICS_DIR = f'results/metrics/{MODEL}'
FIG_DIR = f'results/figures/{MODEL}'
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

stages = {}
for stage in ['S0', 'S1', 'S2', 'S3', 'S4', 'S5']:
    p = f'{PRED_DIR}/{stage}.json'
    if os.path.exists(p):
        with open(p) as f:
            stages[stage] = json.load(f)
        print(f'Loaded {stage}: {len(stages[stage])} records')
    else:
        print(f'MISSING: {stage}')


## 1. Accuracy table

In [ ]:
acc_table = {s: accuracy_per_task(rs) for s, rs in stages.items()}
with open(f'{METRICS_DIR}/accuracy_per_task.json', 'w') as f:
    json.dump(acc_table, f, indent=2)

import pandas as pd
df = pd.DataFrame(acc_table).T
print(df.to_string(float_format='%.3f'))


## 2. Confidence summaries

In [ ]:
conf_table = {}
ece_table = {}
for s, rs in stages.items():
    conf_table[s] = {
        'verbalized': verbalized_confidence_stats(rs),
        'logprob': logprob_confidence_stats(rs),
    }
    ece_table[s] = {
        'logprob': expected_calibration_error(rs, source='logprob'),
        'verbalized': expected_calibration_error(rs, source='verbalized'),
    }

with open(f'{METRICS_DIR}/confidence.json', 'w') as f:
    json.dump(conf_table, f, indent=2, default=str)
with open(f'{METRICS_DIR}/ece.json', 'w') as f:
    json.dump(ece_table, f, indent=2, default=str)

print('ECE per stage:')
for s, vals in ece_table.items():
    print(f'  {s}: logprob={vals["logprob"]:.3f}  verbalized={vals["verbalized"]:.3f}'
          if vals['logprob'] is not None and vals['verbalized'] is not None else f'  {s}: --')


## 3. Confidence Drop under modality ablation

In [ ]:
if 'S3' in stages:
    drops = {}
    if 'S2' in stages:
        drops['audio_removal_S3_minus_S2'] = confidence_drop(stages['S3'], stages['S2'], source='verbalized')
    if 'S1' in stages:
        drops['visual_removal_S3_minus_S1'] = confidence_drop(stages['S3'], stages['S1'], source='verbalized')
    with open(f'{METRICS_DIR}/confidence_drops.json', 'w') as f:
        json.dump(drops, f, indent=2)
    print(json.dumps(drops, indent=2))


## 4. Attribution Faithfulness Score (HEADLINE METRIC)

In [ ]:
if {'S1', 'S2', 'S3'}.issubset(stages):
    afs = attribution_faithfulness(stages['S3'], s2_records=stages['S2'], s1_records=stages['S1'])
    with open(f'{METRICS_DIR}/attribution_faithfulness.json', 'w') as f:
        json.dump(afs, f, indent=2)
    print('Attribution Faithfulness Score per task:')
    for task, d in afs.items():
        if d.get('score') is not None:
            print(f'  {task}: {d["score"]:.2f}  (faithful={d["faithful"]}  confab={d["confabulated"]}  unparsed={d["unparseable"]})')


## 5. Transcript Injection Bias and Lexical Override Rate

In [ ]:
if 'S3' in stages and 'S4' in stages:
    tib = transcript_injection_bias(stages['S3'], stages['S4'])
    with open(f'{METRICS_DIR}/transcript_injection_bias.json', 'w') as f:
        json.dump(tib, f, indent=2)
    print('TIB (positive = transcripts hurt):')
    for t, v in tib.items():
        print(f'  {t}: {v:+.3f}')

if 'S3' in stages and 'S5' in stages:
    lor = lexical_override_rate(stages['S3'], stages['S5'])
    with open(f'{METRICS_DIR}/lexical_override_rate.json', 'w') as f:
        json.dump(lor, f, indent=2)
    print('\nLOR (fraction of S3-correct samples flipped by mismatched transcript):')
    for t, d in lor.items():
        if d.get('lor') is not None:
            print(f'  {t}: {d["lor"]:.2f}  (flipped={d["flipped"]}  stayed={d["stayed"]})')


## 6. Answer flip rates

In [ ]:
flip_rates = {}
pairs = [('S3','S2'), ('S3','S1'), ('S3','S4'), ('S3','S5'), ('S3','S0')]
for a, b in pairs:
    if a in stages and b in stages:
        flip_rates[f'{a}_vs_{b}'] = answer_flip_rate(stages[a], stages[b])
with open(f'{METRICS_DIR}/answer_flip_rates.json', 'w') as f:
    json.dump(flip_rates, f, indent=2)


## 7. Figures

In [ ]:
# Accuracy heatmap (stages × tasks)
import pandas as pd
df = pd.DataFrame(acc_table).T
df = df.drop(columns=['OVERALL'], errors='ignore')
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(df.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(df.columns))); ax.set_xticklabels(df.columns)
ax.set_yticks(range(len(df.index))); ax.set_yticklabels(df.index)
for i in range(len(df.index)):
    for j in range(len(df.columns)):
        v = df.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9, color='black')
plt.colorbar(im, ax=ax, label='Accuracy')
plt.title(f'{MODEL.upper()} accuracy: stages × tasks')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/accuracy_heatmap.png', dpi=150)
plt.show()


In [ ]:
# AFS bar chart
if {'S1','S2','S3'}.issubset(stages):
    tasks_only = [k for k in afs.keys() if k != 'OVERALL']
    scores = [afs[t]['score'] for t in tasks_only if afs[t]['score'] is not None]
    labels = [t for t in tasks_only if afs[t]['score'] is not None]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, scores, color='steelblue')
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='random / chance')
    ax.set_ylabel('Attribution Faithfulness Score')
    ax.set_title(f'{MODEL.upper()}: AFS by task (lower = more confabulation)')
    ax.set_ylim(0, 1.0)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/afs_by_task.png', dpi=150)
    plt.show()


In [ ]:
# LOR vs TIB scatter — both measure text/audio trust at the per-task level
if 'S4' in stages and 'S5' in stages and 'S3' in stages:
    tasks_in = [t for t in tib if t != 'OVERALL' and lor.get(t, {}).get('lor') is not None]
    fig, ax = plt.subplots(figsize=(6, 5))
    for t in tasks_in:
        ax.scatter(tib[t], lor[t]['lor'], s=80)
        ax.annotate(t, (tib[t], lor[t]['lor']), xytext=(5,5), textcoords='offset points')
    ax.axhline(0, color='gray', alpha=0.3)
    ax.axvline(0, color='gray', alpha=0.3)
    ax.set_xlabel('TIB = Acc(S3) - Acc(S4)  (>0 = transcripts hurt)')
    ax.set_ylabel('LOR = fraction flipped by mismatched transcript')
    ax.set_title(f'{MODEL.upper()}: lexical bias signals by task')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/tib_vs_lor.png', dpi=150)
    plt.show()


## What to look for

- **Headline:** Is overall AFS substantially below 1.0? If yes, this is the confabulation finding. Best if AFS varies by task — e.g., HIGH on AIE (where audio is genuinely needed) and LOW on AVCM (where the model can fake it).
- **Calibration:** Does verbalized confidence saturate near 100? Is logprob ECE > 0.1?
- **Lexical override:** LOR > 0.2 on audio-essential tasks would be strong evidence the model doesn't trust its own audio encoder.
- **TIB shape:** Predicted TIB > 0 for ACC/AEL (audio-essential) and TIB < 0 for AVTM (text-helps).
